# **Fairness-Aware Age Estimation: Bias Detection and Mitigation - Part III**
UB Master in Fundamental Principels of Data Science (2025-2026)

Author: Julio C. S. Jacques Junior

Last modified: Jan, 2026.

---

# **Part III: Goal**

- Use a **custom loss** (without any data augmentation) to address the bias problem.

- In this notebook, we define a **custom loss** based on the inverse frequency of train samples in the training set, considering only the age attribute. As before, **you are expected to define a more creative solution** in which other attributes or strategies are be considered.

- **Requirements:** Carefully review the instructions in Notebooks **Part I** and **Part II**, and run them.


## Checking the pytorch version
 - This notebook was successfully tested on version = 2.9.0+cu126

In [ ]:
import torch
print(torch.__version__)

## Importing required libraries

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import csv
from PIL import Image
from timm import create_model
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import copy
from tqdm import tqdm

# **Downloading the Appa-Real Dataset**
- Please, check the **detailed instructions in notebook Part I**.

In [ ]:
from zipfile import ZipFile

# downloading the data
!wget https://data.chalearnlap.cvc.uab.cat/Colab_MFPDS/2025/appa-real-dataset_v2.zip

with ZipFile('appa-real-dataset_v2.zip','r') as zip:
   zip.extractall()
   print('Data decompressed successfully')

# removing the .zip file after extraction to clean space
!rm appa-real-dataset_v2.zip

# **Mount Google Drive to save the trained model on the cloud**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
# Note, the default path will be: '/content/gdrive/MyDrive/'
# In my case, the final path will be: '/content/gdrive/MyDrive/temp/' as I
# created a '/temp/' folder in my google drive for this purpose.

# **Defining the Data Loader Class**
- In this example, metadata information is loaded but not used. Future implementations can take benefit of it.
- Note that age labels are divided by 100, as detailed in **Part II**.
- The method `__getitem__` **now returns the indices** `idx` of the samples in a batch **so that custom loss can be applied later**.
- Also note that **no data augmentation is applied**. This way, we can easily compare all evalutated models:
  - with **no data augmentation** and **no custom loss** (Part I)
  - **with data augmentation** and **no custom loss** (Part II)
  - **with custom loss** and **no data augmentation** (Part III)


In [ ]:
class AgeEstimationDataset(Dataset):
    def __init__(self, image_dir, csv_file, base_transforms):
        self.image_dir = image_dir
        self.data_info = pd.read_csv(csv_file)
        self.base_transforms = base_transforms
        self.age_normalization_factor = 100; # used to normalize age labels

    def __len__(self):
        return len(self.data_info)

    def __normalization_factor__(self):
        return self.age_normalization_factor

    def __getitem__(self, idx):
        image_id = f"{self.data_info.iloc[idx, 0]:06d}.jpg"  # Format image ID
        image_path = os.path.join(self.image_dir, image_id)
        image = Image.open(image_path).convert("RGB")  # Load image as RGB

        raw_age = float(self.data_info.iloc[idx, 1])
        # normalizing age labes (by 100) to be between 0 and 1 (assuming 100 is the max age)
        age = raw_age / self.age_normalization_factor
        metadata = self.data_info.iloc[idx, 2:].tolist()  # Extract metadata as list

        image = self.base_transforms(image)

        return image, torch.tensor(age, dtype=torch.float32), idx

# **Defining the base image transformations**

In [ ]:
base_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# **Loading the Train and Validation sets**

In [ ]:
# Create dataset and dataloader (train set):

# train set without data augmentation
dataset_train = AgeEstimationDataset("train_data", "labels_metadata_train.csv", base_transforms)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of train samples: {len(dataloader_train.dataset)}")

# Create dataset and dataloader (validation set):
dataset_valid = AgeEstimationDataset("valid_data", "labels_metadata_valid.csv", base_transforms)
dataloader_valid = DataLoader(dataset_valid, batch_size=32, shuffle=True, num_workers=2)
print(f"Total number of valid samples: {len(dataloader_valid.dataset)}")

# **Loading the pretrained ViT baselone model and adapting it to our problem**

- ViT normally outputs class scores for classification tasks; here, we adapt it for regression by setting **num_classes=1**.

> **Note:** This notebook is intended as a starting point. For your deliverables, avoid making only minor modifications. Instead, explore your creativity and try more substantial improvements.

- **It is also recommended to use the same architecture when comparing results with and without data augmentation, in order to ensure a fair comparison.**

In [ ]:
# Vision Transformer Model for Age Prediction (pretrained on ImageNet)
# https://pytorch.org/vision/main/models/vision_transformer.html
# https://huggingface.co/docs/transformers/main/en//model_doc/vit
class AgeEstimationViT(nn.Module):
    def __init__(self):
        super(AgeEstimationViT, self).__init__()
        self.vit = create_model("vit_base_patch16_224", pretrained=True, num_classes=1) # num_classes=1 as we want to regress a single (age value)
        self.activation = nn.Sigmoid()  # Added Sigmoid activation

    def forward(self, x):
        x = self.vit(x)
        return self.activation(x)  # Apply Sigmoid activation

In [ ]:
# creating the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeEstimationViT().to(device)

# Defining an auxiliary function to plot the training history

In [ ]:
# Function to plot training curves
def plot_training_curves(train_losses, val_losses):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training & Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

# **Defining the Training function**
- Our training code includes **early stopping** and **automatic saving of the best model**. Early stopping monitors the validation loss during training and halts training if the model stops improving for a specified number of epochs, helping to prevent overfitting and save computational resources. At the same time, the model with the lowest validation loss is automatically saved, ensuring that we keep the best-performing version for evaluation or deployment.
- In addition, we now **use a custom loss with pre-computed sample weights** (`sample_weights_torch`).


In [ ]:
def train_model(model, dataloaders, criterion, optimizer, scheduler,
                num_epochs, patience, model_path, sample_weights):

    sample_weights_torch = torch.tensor(sample_weights, dtype=torch.float32, device=device)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    early_stopping_counter = 0
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0

            with tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch+1}") as t:
                for inputs, labels, indices in t:
                    inputs = inputs.to(device)
                    labels = labels.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs).view_as(labels)
                        loss_per_sample = criterion(outputs, labels)

                        # Safe indexing for batch weights
                        if isinstance(indices, list):
                            indices = torch.tensor(indices, dtype=torch.long, device=device)
                        elif isinstance(indices, torch.Tensor):
                            indices = indices.view(-1).long().to(device)
                        else:
                            raise TypeError(f"indices must be list or tensor, got {type(indices)}")

                        batch_weights = sample_weights_torch[indices]

                        # Weighted loss
                        loss = (loss_per_sample * batch_weights).mean()

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    t.set_postfix(loss=loss.item())

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            print(f'{phase} Loss: {epoch_loss:.6f}')

            if phase == 'train':
                train_losses.append(epoch_loss)
            else:
                val_losses.append(epoch_loss)
                scheduler.step(epoch_loss)

                # Early stopping
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    early_stopping_counter = 0
                    print("Saving best model...")
                    torch.save(best_model_wts, model_path)
                else:
                    early_stopping_counter += 1
                    if early_stopping_counter >= patience:
                        print("Early stopping triggered.")
                        model.load_state_dict(best_model_wts)
                        plot_training_curves(train_losses, val_losses)
                        return model

        # Free GPU memory
        torch.cuda.empty_cache()

    model.load_state_dict(best_model_wts)
    plot_training_curves(train_losses, val_losses)
    return model


# **Defining our custom loss**

1) First, we read the age labels (using the train set) from CSV and create 5-year bins

In [ ]:
labels = pd.read_csv('labels_metadata_train.csv')

# Extract ages
ages = labels["age"].values.astype(np.float32)

# Create 5-year bins: 0–4 -> 0, 5–9 -> 1, ... (this will generate 17 bins)
age_bins = (ages // 5).astype(int)

2) Then, we compute sample weights (inverse frequency per bin). This makes rare age ranges more important.

In [ ]:
from collections import Counter

bin_counts = Counter(age_bins)
num_samples = len(age_bins)

# Weight per bin = inverse frequency
bin_weights = {b: num_samples / c for b, c in bin_counts.items()}

# Weight per sample
sample_weights = np.array([bin_weights[b] for b in age_bins], dtype=np.float32)
# normalizing the weights by the mean
sample_weights_n = sample_weights / sample_weights.mean()
sample_weights_torch = torch.tensor(sample_weights_n, dtype=torch.float32, device=device)


# **Training the Model (Now with Custom Loss and No Data Augmentation)**

- In the next cell, **we train our model using the same architecture and hyperparameters as in Part I and Part II**.  
- However, **we now use a custom loss** that incorporates the sample weights defined above.
- Note that the loss is now defined as `nn.MSELoss(reduction="none")`, because the **reduction** (how individual element-wise losses are combined) is applied later (using the mean), after applying the sample weights.



In [ ]:
model_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model_custom_loss.pth"

# model hyperparameters
num_epochs = 50
patience = 10

# No reduction is applied.
# It returns the element-wise squared errors, keeping the same shape as the input.
# One can use it when applying custom weighting.
criterion = nn.MSELoss(reduction="none")

optimizer = Adam(model.parameters(), lr=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=patience)

# data loaders
dataloaders = {"train": dataloader_train, "val": dataloader_valid}  # Assuming split dataset

# train the model
best_model = train_model(model, dataloaders, criterion, optimizer, scheduler, num_epochs, patience, model_filename, sample_weights)

# **Defining an auxiliary function to evaluate the model**
- Check Part I for the details.

In [ ]:
# Function to make predictions on test set and compute MSE
def predict_and_evaluate(model_path, test_dataset, output_zip=None, batch_size=32, output_csv="predictions.csv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AgeEstimationViT().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    predictions = []
    actual_ages = []

    with torch.no_grad():
        for images, labels, metadata in tqdm(test_loader, desc="Predicting"):
            images = images.to(device)
            outputs = model(images).squeeze().cpu().numpy()
            labels = labels.cpu().numpy()

            predictions.extend(outputs * test_dataset.__normalization_factor__())
            actual_ages.extend(labels * test_dataset.__normalization_factor__())

    mae = np.mean(np.abs(np.array(predictions) - np.array(actual_ages)))
    if output_zip is not None:
      print(f"\n=======\nMean Absolute Error on Test Set: {mae:.4f}")
    else:
      print(f"\n=======\nMean Absolute Error on Validation Set: {mae:.4f}")


    # Only create ZIP if output_zip is provided
    if output_zip is not None:
        # Save predictions to CSV without headers
        with open(output_csv, mode='w', newline='') as file:
            writer = csv.writer(file)
            for pred in predictions:
                writer.writerow([pred])

        with ZipFile(output_zip, 'w') as zipf:
            zipf.write(output_csv, os.path.basename(output_csv))
        print(f"Predictions saved to {output_csv} and compressed as {output_zip}")

    return predictions, mae

# **Loading the Saved Model and Making Predictions on the Validation Set**


In [ ]:
model_filename = "/content/gdrive/MyDrive/temp/best_age_estimation_model_custom_loss.pth"

# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_valid,output_zip=None, batch_size=32,output_csv=None)



---



# **Generating the submission fie (on the test set) for our challenge**

- Loading the Saved Model and Making Predictions on the Test Set

- The following cells are generating predictions (and evaluating them) on the **Test set** so that we can create our submission file to be uploaded to our age estimation challenge.

- **Do not evaluate your model on the Test set when defining your model, training strategy, or hyperparameters.** For this, use the Validation set.

In [ ]:
# Create dataset and dataloader (test set):
dataset_test = AgeEstimationDataset("test_data", "labels_metadata_test.csv", base_transforms=base_transforms)
print(f"Total number of test samples: {len(dataset_test)}")

In [ ]:
# Run prediction and compute MAE
predictions, mae = predict_and_evaluate(model_filename, dataset_test, "predictionsViTbaseline_custom_loss.zip")